### CG3 Wrapper
In this notebook, we will try running CG3 command line from Python.

Also to create a CG3 wrapper if possible.

In [48]:
print("Hello")

Hello


In [49]:
!cg3 --help

VISL CG-3 Disambiguator version 1.4.18.13898
Usage: vislcg3 [OPTIONS]

Environment variable:
 CG3_DEFAULT: Sets default cmdline options, which the actual passed options will override.
 CG3_OVERRIDE: Sets forced cmdline options, which will override any passed option.

Options:
 -h, --help                 shows this help
 -?, --?                    shows this help
 -V, --version              prints copyright and version information
     --min-binary-revision  prints the minimum usable binary grammar revision
 -g, --grammar              specifies the grammar file to use for disambiguation
     --grammar-out          writes the compiled grammar in textual form to a file
     --grammar-bin          writes the compiled grammar in binary form to a file
     --grammar-only         only compiles the grammar; implies --verbose
     --ordered              (will in future allow full ordered matching)
 -u, --unsafe               allows the removal of all readings in a cohort, even the last one
 -s,

In [50]:
INPUT_FILE = "../data/practice.txt"
GRAMMAR_FILE = "../data/CG3_rules/practice.cg3"
OUTPUT_FILE = "../data/practice_output.txt"

#### Use cg3 command line to process input, using grammar file

In [51]:
# show input content
!cat {INPUT_FILE}

"<word1>"
	"word_1_1" A B 
	"word_1_2" A C
	"word_1_3" A D
	"word_1_4" A E
"<word2>"
	"word_2_1" X Y
	"word_2_2" Z T
"<word3>"
	"word_3_1" 
"<word4>"
	"word_4_1" 
"<word5>"
	"word_5_1" X Y Z
	"word_5_2" W X Y Z
	"word_5_3" T
"<.>"


In [52]:
# show grammar file
!cat {GRAMMAR_FILE}

# This is an example Constraint Grammar rules file.

# There is no built-in manual yet. Resources:
# http://visl.sdu.dk/cg3.html
# http://groups.google.com/group/constraint-grammar
# http://kevindonnelly.org.uk/2010/05/constraint-grammar-tutorial/

# Firstly, we need to define what tags should be considered sentence delimiters. For this example, only full stop is set as delimiter.
DELIMITERS = "<.>" ;

# We can define sets for common tag clusters
LIST SET_A = A ;
LIST SET_B = B ;
LIST SET_C = C ;
LIST SET_D = D ;
LIST SET_X = X ;
LIST SET_Y = Y ;
LIST SET_Z = Z ;
LIST PUNCT = "." ;
LIST @SET_F = F ;



SECTION
# add tag F, on (TARGET) words have tag A, if next word (1) has tag X
ADD:rule6 @SET_F TARGET SET_A IF (1 SET_X) ;

# add tag G, on words (TARGET) have tag X, NO condition (NO IF)
# MAP or ADD command must have TARGET set
MAP:rule7 (G) TARGET (X); 

# select (prefer) tag C, D, E if next word has tag X
SELECT:rule1 (C D E) IF (1 (X)) ;

# Remove tag B IF next word (1) has tag Y
RE

In [53]:
!cg3 -g {GRAMMAR_FILE} -I {INPUT_FILE}

"<word1>"
	"word_1_3" A D F F
"<word2>"
	"word_2_1" X Y G
	"word_2_2" Z T
"<word3>"
	"word_3_1"
"<word4>"
	"word_4_1"
"<word5>"
	"word_5_1" X Y Z G
	"word_5_3" T
"<.>"



#### Write output to a file

In [54]:
!cg3 -g {GRAMMAR_FILE} -I {INPUT_FILE} -O {OUTPUT_FILE}

In [55]:
# Show output content
!cat {OUTPUT_FILE}

"<word1>"
	"word_1_3" A D F F
"<word2>"
	"word_2_1" X Y G
	"word_2_2" Z T
"<word3>"
	"word_3_1"
"<word4>"
	"word_4_1"
"<word5>"
	"word_5_1" X Y Z G
	"word_5_3" T
"<.>"



### Using Subprocess

In [56]:
import subprocess

In [57]:
command = ["cg3", "--help"]  # display cg3 help
output = subprocess.run(command, capture_output=True, text=True)

In [58]:
print(output.stdout)

VISL CG-3 Disambiguator version 1.4.18.13898
Usage: vislcg3 [OPTIONS]

Environment variable:
 CG3_DEFAULT: Sets default cmdline options, which the actual passed options will override.
 CG3_OVERRIDE: Sets forced cmdline options, which will override any passed option.

Options:
 -h, --help                 shows this help
 -?, --?                    shows this help
 -V, --version              prints copyright and version information
     --min-binary-revision  prints the minimum usable binary grammar revision
 -g, --grammar              specifies the grammar file to use for disambiguation
     --grammar-out          writes the compiled grammar in textual form to a file
     --grammar-bin          writes the compiled grammar in binary form to a file
     --grammar-only         only compiles the grammar; implies --verbose
     --ordered              (will in future allow full ordered matching)
 -u, --unsafe               allows the removal of all readings in a cohort, even the last one
 -s,

#### Now running the CG command and save output to a variable

In [59]:
command = ["cg3", "-g", GRAMMAR_FILE, "-I", INPUT_FILE]  
output = subprocess.run(command, capture_output=True, text=True)
print(output.stdout)

"<word1>"
	"word_1_3" A D F F
"<word2>"
	"word_2_1" X Y G
	"word_2_2" Z T
"<word3>"
	"word_3_1"
"<word4>"
	"word_4_1"
"<word5>"
	"word_5_1" X Y Z G
	"word_5_3" T
"<.>"




### Now running CG3 to process custom input text (in a Python variable)

In [60]:
input_text = """
"<word1>"
	"word_1_1" A B 
	"word_1_2" A C
	"word_1_3" A D
	"word_1_4" A E
"<word2>"
	"word_2_1" X Y
	"word_2_2" Z T
"<word3>"
	"word_3_1" 
"<word4>"
	"word_4_1" 
"<word5>"
	"word_5_1" X Y Z
	"word_5_2" W X Y Z
	"word_5_3" T
"<.>"
"""

In [65]:
command = ["cg3", "-g", GRAMMAR_FILE]  
process = subprocess.Popen(command, 
                           stdin=subprocess.PIPE,
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE,
                           text=True,
                           encoding="utf-8"                        
                          )
output, error = process.communicate(input=input_text)
print("Output:\n", output)
print("-"*10)
print("Error:\n", error)

Output:
 "<word1>"
	"word_1_3" A D F F
"<word2>"
	"word_2_1" X Y G
	"word_2_2" Z T
"<word3>"
	"word_3_1"
"<word4>"
	"word_4_1"
"<word5>"
	"word_5_1" X Y Z G
	"word_5_3" T
"<.>"


----------
Error:
 
